In [1]:
import pandas as pd
import math

In [2]:
BENCHMARK_PATH = "../../data/data set/benchmark.csv"

benchmark = pd.read_csv(BENCHMARK_PATH, dtype=str, keep_default_na=False)

### Load the data

The data from B1 is divided into 3 groups:
- B1A/B1B (byte-exact vs case-folded)
- Genes/chemicals
- Canonical/synomym forms

The matches have been established in the previous step and the resulting files have been created. The goal is to extract useful information from this data.

The file system corresponds to the 4x2 grid resulting from the division (8 files in total):

| | Chemicals (canonical) | Chemicals (synonyms) | Genes (canonical) | Genes (synonyms) |
|--|--|--|--|--|
| B1a (exact byte-by-byte) | _b1a_chem_canonical.csv_ | _b1a_chem_synonyms.csv_ | _b1a_gene_canonical.csv_ | _b1a_gene_synonym.csv_ |
| B1b (case-folded) | _b1b_chem_canonical.csv_ | _b1b_chem_synonym.csv_ | _b1b_gene_canonical.csv_ | _b1b_gene_synonym.csv_ |

Each key has been matched to the database according to the rules that follows from its file name.

In [3]:
B1_PATH = "../results"
suffixes = ["a", "b"]
types = ["chem", "gene"]
tiers = ["canonical", "synonym"]
b1_data = [
    {
        'suffix': suffix,
        'type': type_name,
        'tier': tier,
        'df': pd.read_csv(f"{B1_PATH}/b1{suffix}_{type_name}_{tier}.csv", dtype=str, keep_default_na=False)
    }
    for suffix in suffixes
    for type_name in types
    for tier in tiers
]

# First, we assert the data has the correct expected row numbers
genes = [item for item in b1_data if item['df'].shape[0] == 157 and item['type'] == "gene"]
chemicals = [item for item in b1_data if item['df'].shape[0] == 162 and item['type'] == "chem"]
assert len(genes) == 4 and len (chemicals) == 4, "The files do not have the expected row count. Data might be corrupted"

# Next, we check all items have a corresponding entry in the benchmark
benchmark_gene_ids = set(benchmark[benchmark["entity_type"] == "gene"]["id"])
benchmark_chemical_ids = set(benchmark[benchmark["entity_type"] == "chemical"]["id"])

for gene in genes:
    assert set(gene['df']['id']) == benchmark_gene_ids, "Not all gene IDs match the benchmark"

for chemical in chemicals:
    assert set(chemical['df']['id']) == benchmark_chemical_ids, "Not all chemical IDs match the benchmark"

# Now, we check check the expected columns are present

for item in b1_data:
    required_columns = {"id", "input", "n_candidates", "returned_id"}
    if item['type'] == "gene":
        required_columns.add("matched_field")
    missing = required_columns - set(item['df'].columns)
    assert not missing, f"b1{item['suffix']}_{item['type']}_{item['tier']}: missing columns {missing}"

# Fourth, we join on the IDs and check that no inputs have changed
gold_columns = ["id", "input", "canonical_name", "entity_type", "noise_type", "id_status", "accepted_id"]

for item in b1_data:
    prediction_columns = ["id", "input", "n_candidates", "returned_id"]
    if item['type'] == "gene":
        prediction_columns.append("matched_field")

    item['merged'] = pd.merge(
        left=item['df'][prediction_columns],
        right=benchmark[gold_columns],
        on="id",
        how="left",
        suffixes=("_result", "_benchmark"),
        validate="one_to_one"
    )
    assert (item['merged']['input_result'] == item['merged']['input_benchmark']).all(),  f"b1{item['suffix']}_{item['type']}_{item['tier']}: inputs different from benchmark.csv, result file is stale"


# Fifth, we check our policy is implemented correctly; n_candidates converts to int, is >=0 and returned_id is not empty iff n_candidates == 1

for item in b1_data:
    item['merged']['n_candidates'] = item['merged']['n_candidates'].astype(int)
    for row in item['merged'].itertuples():
        assert row.n_candidates >= 0, f"Negative n_candidates on row {row.id}"
        if row.n_candidates == 1:
            assert row.returned_id != "", f"One candidate but no returned id on row {row.id}"
        else:
            assert row.returned_id == "", f"returned_id despite not exactly one candidate on row {row.id}"

In [4]:
# Decision function
# Each item  in the table will get one of six possible outcomes:
# returned_on_none: id_status was none, but returned_id still wasn't empty
# abstained_on_none: id_status was none and returned_id correctly came back empty
# abstain_no_match: id_status wasn't none, but no matches were found
# abstain_ambig: id_status wasn't none, there was more than one candidate
# correct: id_status wasn't none, there was exactly one condidate and it was in the accepted_id list
# wrong_entity: if all else fails, the classification was wrong to begin with
def b1_decide(row):
    if row['id_status'] == "none":
        if row['returned_id'] != "":
            return "returned_on_none"
        return "abstained_on_none"
    
    if row['n_candidates'] == 0:
        return "abstain_no_match"
    
    if row['n_candidates'] > 1:
        return "abstain_multiple"
    if row['returned_id'] in row['accepted_id'].split(";"):
        return "correct"
    return "wrong_entity"

In [5]:
# Create a new data frame to store all of the 8 tables concatenated into one
dfs = []
for item in b1_data:
    df = item['merged'].copy()
    df['case'] = item['suffix']
    df['tier'] = item['tier']
    df['outcome'] = df.apply(b1_decide, axis=1)
    dfs.append(df)

b1_full = pd.concat(dfs, ignore_index=True)
assert len(b1_full) == 4 * 157 + 4 * 162

In [6]:
# Check
print(pd.crosstab([b1_full['entity_type'], b1_full['case'], b1_full['tier']], b1_full['outcome']))

outcome                     abstain_multiple  abstain_no_match  \
entity_type case tier                                            
chemical    a    canonical                 0               149   
                 synonym                   2               107   
            b    canonical                 0               130   
                 synonym                   2                57   
gene        a    canonical                 0               149   
                 synonym                   2                99   
            b    canonical                 0               129   
                 synonym                   3                77   

outcome                     abstained_on_none  correct  returned_on_none  \
entity_type case tier                                                      
chemical    a    canonical                 10        3                 0   
                 synonym                    9       38                 1   
            b    canonical         

In [7]:
# Number of successful resolutions
# Quote: Wilson (1927), the original: E. B. Wilson, "Probable inference, the law of succession, and statistical inference," Journal of the American Statistical Association 22(158), 209–212.

def wilson_score_interval(ups, total, z=1.96): # z=1.96 -> Calculate the 95% confidence interval
    if total == 0:
        return 0

    p = ups / total
    denominator = 1 + z**2 / total
    center = (p + z**2 / (2 * total)) / denominator
    half_w = z * math.sqrt(p * (1 - p) / total + z**2 / (4 * total**2)) / denominator
    return (max(0.0, center - half_w), min(1.0, center + half_w))


In [8]:
id_status_single = b1_full[b1_full['id_status'] == "single"]

grouped = id_status_single.groupby(['entity_type', 'case', 'tier'])

for keys, group in grouped:
    entity_type, case, tier = keys

    ups = (group['outcome'] == "correct").sum()
    total = len(group)
    bottom, top = wilson_score_interval(ups, total)

    print(
        f"b1{case}_{entity_type}_{tier}: "
        f"Success rate: {ups}/{total} [{bottom:.3f}, {top:.3f}]"
    )

b1a_chemical_canonical: Success rate: 3/108 [0.009, 0.079]
b1a_chemical_synonym: Success rate: 30/108 [0.202, 0.369]
b1b_chemical_canonical: Success rate: 14/108 [0.079, 0.206]
b1b_chemical_synonym: Success rate: 67/108 [0.526, 0.706]
b1a_gene_canonical: Success rate: 0/119 [0.000, 0.031]
b1a_gene_synonym: Success rate: 41/119 [0.265, 0.434]
b1b_gene_canonical: Success rate: 11/119 [0.052, 0.158]
b1b_gene_synonym: Success rate: 53/119 [0.359, 0.535]


## Headline table

In [9]:
# Accuracy = correct / n
# Coverage = (correct + wrong) / n
# Precision = correct / (correct + wrong)
headline_table = {
    "Entity type": [key[0] for key, group in grouped],
    "Case": [("exact" if key[1] == "a" else "folded") for key, group in grouped],
    "Tier": [key[2] for key, group in grouped],
    "n": [len(group) for keys, group in grouped],
    "Correct": [(group['outcome'] == "correct").sum() for key, group in grouped],
    "Accuracy": [round((group['outcome'] == "correct").sum()/len(group), 3) for key, group in grouped],
    "95% CI": [tuple(round(x, 3) for x in wilson_score_interval((group['outcome'] == "correct").sum(), len(group))) for key, group in grouped],
    "Coverage": [round(group['outcome'].isin(["correct", "wrong_entity"]).sum() / len(group), 3) for key, group in grouped],
    "Precision": [round((group['outcome'] == "correct").sum() / group['outcome'].isin(["correct", "wrong_entity"]).sum(), 3) for key, group in grouped],
    "Wrong entity": [(group['outcome'] == "wrong_entity").sum() for key, group in grouped]
    }

headline_table = pd.DataFrame(data=headline_table)
headline_table

/tmp/ipykernel_971/136641757.py:13: RuntimeWarning: invalid value encountered in scalar divide
  "Precision": [round((group['outcome'] == "correct").sum() / group['outcome'].isin(["correct", "wrong_entity"]).sum(), 3) for key, group in grouped],


,Entity type,Case,Tier,n,Correct,Accuracy,95% CI,Coverage,Precision,Wrong entity
0,chemical,exact,canonical,108,3,0.028,"(0.009, 0.079)",0.028,1.0,0
1,chemical,exact,synonym,108,30,0.278,"(0.202, 0.369)",0.278,1.0,0
2,chemical,folded,canonical,108,14,0.130,"(0.079, 0.206)",0.130,1.0,0
3,chemical,folded,synonym,108,67,0.620,"(0.526, 0.706)",0.620,1.0,0
4,gene,exact,canonical,119,0,0.000,"(0.0, 0.031)",0.000,NaN,0
5,gene,exact,synonym,119,41,0.345,"(0.265, 0.434)",0.345,1.0,0
6,gene,folded,canonical,119,11,0.092,"(0.052, 0.158)",0.092,1.0,0
7,gene,folded,synonym,119,53,0.445,"(0.359, 0.535)",0.445,1.0,0


## Table grouped by noise type

In [10]:
# Filter synonym and single rows
synonym_data = b1_full[(b1_full['id_status'] == "single") & (b1_full['tier'] == "synonym")]

grouped = synonym_data.groupby(['entity_type', 'noise_type'])

def calc_accuracy_ci(group, case):
    filtered = group[group['case'] == case]
    correct_count = (filtered['outcome'] == "correct").sum()
    low, high = wilson_score_interval(correct_count, len(filtered))
    return f"{correct_count}/{len(filtered)} {100 * correct_count / len(filtered):.1f} [{100 * low:.1f}–{100 * high:.1f}]"

def calc_accuracy(group, case):
    filtered = group[group['case'] == case]
    return (filtered['outcome'] == "correct").sum() / len(filtered)

# Build the table data
table_rows = []
for key, group in grouped:
    assert len(group[group['case'] == "a"]) == len(group[group['case'] == "b"])
    entity_type, noise_type = key
    exact = calc_accuracy_ci(group, "a")
    folded = calc_accuracy_ci(group, "b")
    gain = round(100 * (calc_accuracy(group, "b") - calc_accuracy(group, "a")), 1)
    table_rows.append([entity_type, noise_type, exact, folded, gain])

class_table = pd.DataFrame(table_rows, columns=["Entity type", "Noise type", "Exact", "Folded", "Fold gain (pp)"])

class_table


,Entity type,Noise type,Exact,Folded,Fold gain (pp)
0,chemical,abbreviation,10/13 76.9 [49.7–91.8],11/13 84.6 [57.8–95.7],7.7
1,chemical,casing,1/13 7.7 [1.4–33.3],13/13 100.0 [77.2–100.0],92.3
2,chemical,separator,1/12 8.3 [1.5–35.4],9/12 75.0 [46.8–91.1],66.7
3,chemical,synonym,13/25 52.0 [33.5–70.0],22/25 88.0 [70.0–95.8],36.0
4,chemical,tool_artifact,0/10 0.0 [0.0–27.8],0/10 0.0 [0.0–27.8],0.0
5,chemical,typo,0/18 0.0 [0.0–17.6],1/18 5.6 [1.0–25.8],5.6
6,chemical,unperturbed,5/7 71.4 [35.9–91.8],7/7 100.0 [64.6–100.0],28.6
7,chemical,word_order,0/10 0.0 [0.0–27.8],4/10 40.0 [16.8–68.7],40.0
8,gene,abbreviation,20/21 95.2 [77.3–99.2],20/21 95.2 [77.3–99.2],0.0
9,gene,casing,0/11 0.0 [0.0–25.9],11/11 100.0 [74.1–100.0],100.0


In [11]:
wrong_rows = b1_full[b1_full['outcome'].isin(["wrong_entity", "returned_on_none"])]
wrong_rows = wrong_rows[["id", "case", "tier", "input_benchmark", "entity_type", "noise_type", "id_status",
                        "matched_field", "returned_id", "accepted_id", "outcome"]]
wrong_rows = wrong_rows.sort_values(["id", "tier", "case"])
#wrong_rows.to_csv(f"{B1_PATH}/wrong_rows/wrong_rows.csv")
wrong_rows

,id,case,tier,input_benchmark,entity_type,noise_type,id_status,matched_field,returned_id,accepted_id,outcome
827,CHEM-028,b,synonym,DHA,chemical,abbreviation,ambiguous,NaN,15608515,445580,wrong_entity
190,CHEM-029,a,synonym,EPA,chemical,abbreviation,ambiguous,NaN,446284,5282847,wrong_entity
828,CHEM-029,b,synonym,EPA,chemical,abbreviation,ambiguous,NaN,446284,5282847,wrong_entity
192,CHEM-031,a,synonym,G6P,chemical,abbreviation,ambiguous,NaN,439284,439958;99058,wrong_entity
830,CHEM-031,b,synonym,G6P,chemical,abbreviation,ambiguous,NaN,439284,439958;99058,wrong_entity
201,CHEM-040,a,synonym,PBS,chemical,abbreviation,ambiguous,NaN,14819,24978514,wrong_entity
839,CHEM-040,b,synonym,PBS,chemical,abbreviation,ambiguous,NaN,14819,24978514,wrong_entity
697,CHEM-060,b,canonical,Alpha-ketoglutarate,chemical,casing,ambiguous,NaN,164533,51,wrong_entity
859,CHEM-060,b,synonym,Alpha-ketoglutarate,chemical,casing,ambiguous,NaN,164533,51,wrong_entity
722,CHEM-085,b,canonical,vitamin B6,chemical,synonym,ambiguous,NaN,104817,1054,wrong_entity


In [12]:
ambig_rows = b1_full[b1_full['id_status'] == "ambiguous"]
pd.crosstab([ambig_rows['entity_type'], ambig_rows['case'], ambig_rows['tier']],
            ambig_rows['outcome'])

outcome                     abstain_multiple  abstain_no_match  correct  \
entity_type case tier                                                     
chemical    a    canonical                 0                44        0   
                 synonym                   1                30        8   
            b    canonical                 0                36        3   
                 synonym                   1                17       16   
gene        a    canonical                 0                30        0   
                 synonym                   1                22        2   
            b    canonical                 0                21        9   
                 synonym                   1                13       11   

outcome                     wrong_entity  
entity_type case tier                     
chemical    a    canonical             0  
                 synonym               5  
            b    canonical             5  
                 synonym              10  
gene        a    canonical             0  
                 synonym               5  
            b    canonical             0  
                 synonym               5

Major findings about baseline B1:
- Accuracy is generally low. Its high precision comes from low coverage. Exact string matching isn't a reliable normalization technique
- Exact matching is precise but brittle. All answers on "single" rows are correct, its failure come mainly from abstentions rather than errors
- Precision remained at 100% throughout. Case-folding did **not** cost precision
- Case-folding helped chemicals far more than genes
- The biggest improvement came from including synonyms in the vocabulary. The step from the canonical -> synonym tiers was the biggest improvement for both chemicals and genes
- Even though the baseline performed exact matching and fabrication was impossible, wrong entities can still be returned
    - KARS/KRAS: The typo itself matched a different gene and therefore got wrongfully matched
    - Competing referents (alias/prev_symbol): An entity may have been matched to an alias or a previous symbol (e.g. COX-2 -> PTGER2)
    - Formula parsing: Pubchem in some cases parsed a wrong name as a chemical formula (e.g. N -> nitrogen, PbS -> lead sulfide)
    - Related compounds: Underspecified inputs lead to matching with sibling records (e.g. Lactate -> stereo-unspecified lactate, G6P -> $\alpha$-G6P)